# Neural Networks for Regression

In this notebook, we're going to be using a neural network to perform regression between a set of input variables and a target variable. Our inputs will be reflectance values from the ESA OC-CCI archive, and our target will be Chlorophyll concentration. These are all continuous variables.

This tutorial isn't going to explain the structure and function of Artificial Neural Networks (ANNs) in detail - there are much better resources for that! What you need to know is that neural networks allow us to model complex non-linear relationships between variables. All ANNs are made up of an input layer (our predictors or *x* values), a number of hidden intermediate layers, and an output layer (our target or *y* value).

![Diagram of a Neural Network](https://github.com/NEODAAS/ML4EO26/blob/practical_2/images/ANN_annot.png?raw=1)

Every layer is made up of **neurons**, with **vertices** connecting the nodes to each other. Every neuron has can accept multiple inputs, and produces an output based on the sum of the values of the input vertices (known as **weights**) plus a **bias** term. This value is then passed through an **activation function**, which is usually non-linear, to produce the node's output. It is the weights and biases of the model that we need to tweak to make the model produce the correct output for any given input.

In the last example, we trained a Random Forest classifier to predict land cover class based on surface reflectance values from the Landsat instruments. ANNs, like other machine learning algorithms, can be trained to predict the correct output given an input. During training, we "show" the network examples for which we already know the target. These examples are used to incrementally update the weights and biases of the model, so that it gradually improves. This improvement is achieved through an optimisation algorithm called **stochastic gradient descent**. A **loss function**, for example Mean Squared Error (MSE), is used to evaluate the model, with the objective of minimizing the loss.

Here are some great resources if you want to learn more about ANNs:

- [But What is a Neural Network?](https://www.youtube.com/watch?v=aircAruvnKk)
- [Dive Into Deep Learning](https://d2l.ai/chapter_preface/index.html)
- [Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/)

For now, you should understand that ANNs are made up of layers, which in turn are made up of neurons. These layers are connected by vertices such that the outputs of each layer directly feed in to the inputs of the next. The objective of training a neural network is to increase the accuracy and decrease the loss (error).

Firstly, we need to select a `GPU` runtime this time, so that we can use the GPU to train our neural network faster.

Select in the top right corner, where the notebook says `Connect` and `Change Runtime Type` and select one of the `GPU` option, such as `T4 GPU`.

In [ ]:
%pip install netCDF4 xarray[io]

Now we need to restart the runtime, to ensure that our installed. So go to the `Runtime` menu at the top and select `Restart Session`. Then we should have everything set up.

## Fetching the data

OK, let's move on to loading some data.

The NERC Earth Observation Data Analysis and AI Service (NEODAAS) hosts a large archive of satellite data, which has been preprocessed to "Level 3". You can examine some of this data in our [portal](https://data.neodaas.ac.uk/cruise/). NEODAAS infastructure is also used to host data for other projects, such as the ESA Ocean Colour Climate Change Initiative (OC-CCI) which is made available via and openDAP server, which we can query directly using Xarray.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib as mpl
from matplotlib import pyplot as plt

import tensorflow as tf

In [ ]:
tf.config.list_physical_devices('GPU')

Firstly, check that we can see `device_type='GPU'` as the output of the above cell - this confirms that the correct kernel is properly installed and selected. Let us know if there are any issues with this, so we can fix it before you go to far.


In [ ]:
# Set up catalogue. For this example use CCI V6.0 1km Daily global data
opendap_url = f"https://rsg.pml.ac.uk/thredds/dodsC/CCI_ALL-v6.0-1km-DAILY"

data = xr.open_dataset(opendap_url, engine="netcdf4")

In [ ]:
# #If we have a problem with the remote server we can download this file instead
# !wget http://data.neodaas.ac.uk/ml4eo/practical_2.nc
# file_path = f"practical_2.nc"
# data = xr.open_dataset(file_path, engine="netcdf4")

Next we're going to fetch the data and load it into an Xarray dataset. You should be familiar with these from the first tutorial.

It's quite a large dataset, so this will take a minute or two to load.

Let's have a look at the dataset. You can expand and contract the data variables and attributes sections to look at the data in more detail.

In [ ]:
data

This is quite a big dataset and we don't need all of it for this tutorial. Let's subset a single time point and reduce the area a bit.

In [ ]:
subset = data.sel(
    time=["2022-08-10"], lat=slice(51, 48), lon=slice(-12, 3),
)
subset

In [ ]:
filename = "practical_2.nc"

subset.to_netcdf(
        filename,
        encoding={v:{'zlib':True, 'complevel':5} for v in subset.data_vars}
    )


That's much smaller. Let's try plotting some of this data.

In [ ]:
subset["chlor_a"].plot()

That doesn't look very clear, lets plot chl on a log scale

In [ ]:
subset["chlor_a"].plot(norm=mpl.colors.LogNorm())

The dataset also contains a lot of metadata about each variable. We can have a look at a specific variable and its attributes.

In [ ]:
subset["Rrs_443"]

Our input variables in this case are going to be the sea surface reflectance bands, which are all prefixed with **Rrs_**. We can extract just these variables from the dataset, along with the **chlor_a** variable, which is our target variable (Chlorophyll concentration).

In this case, Chlorophyll concentration is actually derived from the sea surface reflectance bands using the chlor_a algorithm, which is why they are distributed together. So our model is actually going to be trying to emulate the results of this algorithm.

In [ ]:
var_names = [
    x for x in subset.variables.keys() if x.startswith("Rrs_")
]
var_names.append("chlor_a")

dataset = subset[var_names]
dataset

As in the last tutorial, we need the data to be in a tabular format rather than a raster format, so we're going to convert it into a Pandas dataframe.

In [ ]:
dataframe = dataset.to_dataframe()

Now we're going to tidy the data up a bit, first by dropping the time, latitude, and longitude variables, and then by removing NaN values. NaNs are a specific value that indicates missing data, and we can't include these when training the network; they will cause our weights to explode and prevent the network from training. If we wanted to maintain spatial cohesion, we could keep a record of these missing values like we did with cloud contaminated pixels in the last tutorial. However, in this case we're going to just drop them.

In [ ]:
dataframe = dataframe.reset_index(drop=True)
dataframe = dataframe.dropna()  # Drop NaN values
dataframe

## Building the model

Now we have our training data, it's time to construct the model. We're going to do this using the [Keras](https://keras.io/getting_started/) and [Tensorflow](https://keras.io/getting_started/) libraries.

And verify that tensorflow was properly installed with the GPU

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.models import Model

The following cell contains our model architecture. This is build using something called the Keras Functional API. We create a set of layers, each of which is connected to the previous layer.

The **Input** layer is self-explanatory - this is the data we're providing to the network as inputs. We have 6 input variables, so the size of this layer is 6.

**Dense** layers consist of *n* neurons, each of which is fully connected to the previous layer (meaning that every neuron in the layer recieves input from every neuron in the previous layer). In this case *n*=16. This is a very small network.

At the end we compile the model, using MAE as the loss and the Adam optimizer. Adam is generally a good optimizer for regression problems.

In [ ]:
def build_model():

    input_layer = Input(shape=(6,))

    dense1 = Dense(8, activation=tf.nn.relu)(input_layer)
    dense2 = Dense(8, activation=tf.nn.relu)(dense1)

    output_layer = Dense(1)(dense2)

    model = Model(inputs=input_layer, outputs=output_layer)

    opt = tf.keras.optimizers.Adam(
        learning_rate=0.01
    )  # 0.01 is the default learning rate

    model.compile(loss="mse", optimizer=opt)

    return model

We can have a look at our model structure using the **summary** method.

You might see a lot of output from running this cell; check that the last line says "Created TensorFlow device". This indicates that the cell ran successfully.

In [ ]:
model = build_model()
model.summary()

This tells us that the model around 150 trainable parameters - this is very small for an ANN.

## Pre-processing

Before we train the model, we need to do three things. First, we should shuffle our dataset so that the rows are in random order. Since we know that our dataset is spatially correlated, this is important to ensuring that our data is distributed equally when we split it data later on.

We're going to do this by randomly sampling the dataframe.

In [ ]:
dataframe = dataframe.sample(frac=1)

Next, we need to separate out our input and output variables - currently they're all part of the same dataframe. We can easily extract our **y** values using the pop() method. This removes the **chlor_a** column from the dataframe and stores it in a new variable.

In [ ]:
y = dataframe.pop("chlor_a")

Now we're going to split the dataset into **training** and **test** sets. The training set will be used to train the model. The test set will only be used to assess the performance of our our final, trained model. The training set will actually be split again later on create a **validation** set as well, which will be used to assess the model during training.

Initially, we create the training and test sets by splitting the data 80/20. To do this we will use the same scikit-learn function that we introduced in the first tutorial. Here we're also setting the *random_state* to 456; this will ensure we get the same split every time. This is useful when you're frequently re-running the same script and need reproducible results.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    dataframe, y, train_size=0.8, shuffle=True, random_state=456
)

In [ ]:
print("Number of training samples: {}".format(len(x_train)))
print("Number of test samples: {}".format(len(x_test)))

We have hundreds of thousands of samples - this is a substantial amount of data. The more data you can provide to your model, the better it will perform on unseen data. Your training set should aim to be as representative as possible as the underlying distribution it is sampled from.

Finally, we're going to normalise our input variables. This is because the values for each band lie within quite different scales. We can see this by using the *describe()* method (note that the first row is the count, which is why it's such a big number). Normalisation prevents variables with larger values having a greater influence on the outcome than variables with smaller values. There's more information on this [here](https://towardsai.net/p/data-science/how-when-and-why-should-you-normalize-standardize-rescale-your-data-3f083def38ff).

In [ ]:
x_train.describe()

Scikit-learn has a handy function we can use for normalisation. In this case, we're going to normalise our data so that each variable is on a 0-1 scale.

Be careful not to run this cell twice - you don't want to normalise data that's already been normalised.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
x_train[x_train.columns] = scaler.fit_transform(x_train[x_train.columns])

## Training the model

Next we're going to train the model. Training a neural network involves many different hyperparameters, all of which can be tuned to optimise training. Two important parameters are the **batch size** and the number of **epochs** to train for. A batch (also called a mini-batch) is the number of samples fed through the network before it updates. Batches are used because they help to prevent overfitting - if we showed the model all of the data at once, it could work out exactly what values would produce the correct output every time, but it would struggle to generalise to new data. By only showing it a bit of data at once, we prevent the model from becoming too attuned to the training set. An epoch is one whole pass over all of the batches.

Have a look at [this article](https://machinelearningmastery.com/difference-between-a-batch-and-an-epoch/) for more details on batches and epochs.

We'll start off with a batch size of 128 and train over 10 epochs. Typically you want the batch size to be in the order of a few tens to a few hundreds of samples; very low batch sizes will mean the model takes a long time to train, but large batch sizes increase the risk of overfitting.

We're also going to make use of a built-in ability to divide our data into a training and a validation set. So, our training set is going to be split so that 80% is used for actually training the model, and 20% is used to evaluate it while it's training. This means that after each epoch, Keras will use the model to predict the *y* values from the validation set, and use them to calculate the validation loss (error).

In [ ]:
batch_size = 128
epochs = 10

Given the size of the dataset, training will take a few minutes. You should see the training and validation loss decrease over time. Typically, validation loss will be higher throughout than training loss, because the model will always perform slightly worse on unseen data.

In [ ]:
# Train the model - this could take a while
history = model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,
    verbose=1,
)

We can plot our loss over time to get a better idea of how well the model is training.

In [ ]:
pd.DataFrame(history.history)[["loss","val_loss"]].plot()
plt.title("Model loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")

Model training is a stochastic process and the loss will look different every time you train the model (and for everyone in the workshop). However, you're probably seeing that the training loss is relatively smooth and decreases over time, but the validation loss jumps around a bit. The validation loss might even be lower than the training loss at times.

Ideally, loss for the validation set would always be higher than for the training set, because the model should perform worse on unseen data. But this is not always the case. Neural networks use a process called **stochastic gradient descent** to update the model weights and biases. The nature of gradient descent means that the model will not always take the optimal path to the best solution. There's a good explanation of gradient descent in [this video](https://www.youtube.com/watch?v=IHZwWFHWa-w).

In addition, we only trained our model over 10 epochs. In a real use-case, we would train over hundreds or even thousands of epohs, and small fluctuations would become less important.

Let's have a look at how well our model performs by making predictions and comparing them to our actual *y* values.

In [ ]:
results = model.predict(x_train)

We can plot our predicted vs. actual values to visualise model performance on the training set. We're going to import the Numpy library, which we can use to fit a first-order polynomial model to our data.

In [ ]:
fig, axs = plt.subplots(figsize=(10, 7))

# Plot actual vs. predicted values
axs.scatter(y_train, results, s=1)
axs.set_ylabel("Predicted")
axs.set_xlabel("Actual")

# Use Numpy to fit a linear model using Ordinary Least Squares
z = np.polyfit(y_train, results, 1)

# Plot the model for our predicted data
p = np.poly1d(np.squeeze(z))
axs.plot(y_train, p(y_train), "r--", label="Model")

# Plot the target line
axs.plot((0, y_train.max()), (0, y_train.max()), label="Target")

axs.legend()

It looks as though our model is underestimating the true *y* values. To get a better result, we can carry out **hyperparameter tuning**. Hyperparameters are the model parameters we have control over. These include:

- **The number, type, and size of layers.**
- **The activation functions.**
- **The loss metric.**
- **The batch size.**
- **The number of epochs.**
- **The optimizer (Adam in this case).**
- **The learning rate.**

**Use the cells below to try building and training your own model. You can start by copying the build_model() function above.**

**Try altering some of the hyperparameters and re-training the model. How does the result change?**

You might find it helpful to look at [the Keras documentation on optimizers](https://keras.io/api/optimizers/).

In [ ]:
def build_model():

    # Build your model here

    return model


# Build and train the model
model = build_model()
history = model.fit(
    x_train, y_train, epochs=10, batch_size=128, validation_split=0.2, verbose=1
)

In [ ]:
# Use this cell to plot you model's loss over time

## Callbacks

Earlier, we mentioned that training the model for longer could be a good way to improve its accuracy. However, what if this takes hours or even days? This introduces a risk that training will be interrupted and your model's progress will be lost.

One way to mitigate this is to save our model weights as we go using something called a **callback**. Callbacks are special functions that are executed during model training (usually at specified stages). **Checkpointing**, or saving a snapshot of our model at a specific point in time, is a common use for callbacks. It's easy to insert a callback when calling model.fit().

The following callback will save the model weights every time the validation loss improves (reduces).

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

filepath = "weights-improvement-{epoch:02d}-{val_loss:.2f}.keras"  # This will save the epoch number and validation loss in the file name
checkpoint = ModelCheckpoint(
    filepath, monitor="val_loss", verbose=1, save_best_only=True, mode="min"
)

As the model trains, you should see some files appearing in the sidebar on the left.

In [ ]:
model = build_model()
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=batch_size,
    validation_split=0.2,
    verbose=1,
    callbacks=[checkpoint],
)

We could also just overwrite the saved weights if the model improves, by changing *filepath* to a fixed rather than a variable name, e.g. "best_weights.hdf5".

**Complete the cell below so that the model weights are overwritten as the model trains.**

In [ ]:
filepath = # Complete this line
checkpoint = ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=True, mode='min')

history = model.fit(x_train, y_train, epochs=5, batch_size=batch_size, validation_split = 0.2, verbose=1, callbacks=[checkpoint])

Once we have a set of saved weights, we can load them to continue training if our script is interrupted.

In [ ]:
model = build_model()
model.load_weights(filepath)

You can now continue training the model from where you left off.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=1,
    batch_size=batch_size,
    validation_split=0.2,
    verbose=1,
    callbacks=[checkpoint],
)

Being able to save and load model weights also allows for something called **transfer learning**. This is when we **pre-train** train our model on one dataset, before fine-tuning its performance on another. For many image analysis tasks, models are available that have been pre-trained on an existing image database. This can reduce training time substantially.

For more details on transfer learning, check out [this article](https://www.analyticsvidhya.com/blog/2017/06/transfer-learning-the-art-of-fine-tuning-a-pre-trained-model/).

Another useful callback is **early stopping**. This causes the model to stop training once a metric stops improving.

The following cell implements an early stopping callback which will stop training when two epochs have passed without at least an improvement of 0.05 in the validation loss. Note that we are looking to minimize the loss, so the mode is set to *min*.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss", verbose=1, patience=2, mode="min", min_delta=0.05
)

We can implement multiple callbacks by passing them as a list.

**Complete the cell below, adding the second callback to the list of callbacks passed to model.fit().**

In [ ]:
history = # Add your code here

There are [many other](https://blog.paperspace.com/tensorflow-callbacks/) callbacks you can use in your work.

OK, so we've seen that we can fine-tune our model, and save intermediate outputs. Let's put this together to try training a new model with some different hyperparameters. We'll then apply the model to our hold-out (test) set to evaluate its final performance.

Here's a new model structure with a few changes. Feel free to insert whatever hyperparameters you have found work best for the problem.

In [ ]:
def build_better_model():

    input_layer = Input(shape=(6,))

    dense1 = Dense(512, activation=tf.nn.relu)(input_layer)
    dense2 = Dense(512, activation=tf.nn.relu)(dense1)

    # Dropout layers cause the outputs of some nodes to be randomly 'dropped', i.e.
    # ignored by the next layer. This helps prevent overfitting
    dropout1 = Dropout(0.5)(dense2)  # Apply 50% dropout

    dense3 = Dense(256, activation=tf.nn.relu)(dropout1)
    dense4 = Dense(256, activation=tf.nn.relu)(dense3)

    output_layer = Dense(1)(dense4)

    model = Model(inputs=input_layer, outputs=output_layer)

    opt = tf.keras.optimizers.Adam(learning_rate=0.001)

    model.compile(loss="mse", optimizer=opt)

    return model

Build the model and create some callbacks.

In [ ]:
model = build_better_model()

checkpoint = ModelCheckpoint(
    "much_better_model.keras",
    monitor="val_loss",
    verbose=1,
    save_best_only=True,
    mode="min",
)
early_stopping = EarlyStopping(
    monitor="val_loss", verbose=1, patience=5, mode="min", min_delta=0.005
)

Train the model for 20 epochs.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    verbose=1,
    callbacks=[checkpoint, early_stopping],
)

Once the model is trained, perform prediction on the test set. We need to normalise this first - since the training data was normalised, the test dataset needs to be too.

In [ ]:
x_test[x_test.columns] = scaler.transform(x_test[x_test.columns])

**Complete the cell below to generate predictions for the test set.**

In [ ]:
test_results = # Add your code here

Plot the result.

In [ ]:
fig, axs = plt.subplots(figsize=(10, 7))

# Plot actual vs. predicted values
axs.scatter(y_test, test_results, s=1)
axs.set_ylabel("Predicted")
axs.set_xlabel("Actual")

# Use Numpy to fit a linear model using Ordinary Least Squares
z = np.polyfit(y_test, test_results, 1)

# Plot the model for our predicted data
p = np.poly1d(np.squeeze(z))
axs.plot(y_test, p(y_test), "r--", label="Model")

# Plot the target line
axs.plot((0, y_test.max()), (0, y_test.max()), label="Target")

axs.legend()

Hopefully you see a reasonable result - though remember it will depend a lot on your hyperparameters.

Training ANNs is often a frustrating process and fine-tuning your model can take a long time. We have to find the right balance so that our model does not overfit or underfit, but generalises well to new data. This is why it's important to have a test set to evaluate the final performance of the model.

The entire Neural Network training and learning process is based on an algorithm called Backpropagation, where the derivates of each node are used to identify how to modify the individual values at each point. There is a great [introductory video explaining backpropagation in detail](https://www.youtube.com/watch?v=Ilg3gGewQ5U).

In the next notebook, we will look at a more complex problem: Segmentation.